# 00 — Setup y Hardware 🖥️

> **LLM Lab · Fase 0 — Preparación del laboratorio**
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astorress/llm-lab/blob/main/notebooks/00_setup_y_hardware.ipynb)

Este es el notebook de arranque del laboratorio. Aquí dejamos el entorno listo y entendemos **el hardware sobre el que va a vivir todo lo demás**: qué GPU tenemos, cuánta VRAM podemos gastar, qué precisión numérica soporta y cómo garantizar que cada experimento sea reproducible.

## Objetivos de aprendizaje

Al terminar este notebook vas a poder:

1. **Detectar automáticamente** el hardware disponible (CUDA > MPS > CPU), su VRAM y su soporte de `bfloat16`.
2. **Explicar las limitaciones de Google Colab** (desconexiones, cuotas, fp16 en T4) y cómo las mitigamos en el lab.
3. **Presupuestar la VRAM** de un entrenamiento con la regla de ~16 bytes/parámetro (AdamW + precisión mixta) y aplicarla a los dos modelos del lab (SMALL ~4–5 M y LARGE ~85 M de parámetros).
4. **Controlar las 4 fuentes de aleatoriedad** (Python `random`, NumPy, PyTorch CPU, CUDA/cuDNN) y demostrar reproducibilidad real.
5. **Medir el rendimiento** de tu GPU con un benchmark de multiplicación de matrices (TFLOPS en fp32/fp16) y monitorear la VRAM en vivo.
6. **Seleccionar el perfil** SMALL/LARGE que usan todos los notebooks 01–08.

## Prerrequisitos

- Python básico (funciones, diccionarios).
- Nociones mínimas de PyTorch (qué es un tensor). No se necesita nada más: este es el punto de partida.

## Tiempo estimado

| Perfil | Hardware objetivo | Tiempo |
|--------|-------------------|--------|
| SMALL  | Colab Free (T4) / CPU / Apple Silicon | ~30 min |
| LARGE  | Colab Pro (A100 / L4) | ~30 min |


## Tabla de contenidos

1. [Cómo usar este repo: Colab vs local](#1.-Cómo-usar-este-repo:-Colab-vs-local)
2. [Detección automática de hardware](#2.-Detección-automática-de-hardware)
3. [Limitaciones de Colab](#3.-Limitaciones-de-Colab)
4. [Presupuesto de VRAM](#4.-Presupuesto-de-VRAM)
5. [Seeds y reproducibilidad](#5.-Seeds-y-reproducibilidad)
6. [Selección de perfil SMALL/LARGE](#6.-Selección-de-perfil-SMALL/LARGE)
7. [Benchmark de matmul](#7.-Benchmark-de-matmul)
8. [Monitoreo de VRAM](#8.-Monitoreo-de-VRAM)
9. [Google Drive para checkpoints](#9.-Google-Drive-para-checkpoints)
10. [Resumen](#10.-Resumen)
11. [Ejercicios propuestos](#11.-Ejercicios-propuestos)
12. [Referencias](#12.-Referencias)


## 1. Cómo usar este repo: Colab vs local

Este laboratorio está diseñado para correr **igual** en dos entornos:

| | Google Colab (recomendado) | Local (macOS / Linux) |
|---|---|---|
| **GPU** | T4 gratis · L4/A100 con Pro | La que tengas (o CPU / Apple Silicon) |
| **Setup** | Cero: abrir el notebook y ejecutar | `python -m venv .venv && pip install -e ".[dev]"` |
| **Persistencia** | ❌ el disco se borra al cerrar → usamos Google Drive | ✅ disco local |
| **Perfil típico** | SMALL (T4) o LARGE (A100) | SMALL |

**Regla de oro del lab:** cada notebook es **autocontenido** — corre completo en Colab *sin clonar el repo*. El paquete `src/llm_lab/` es el *espejo* del código de los notebooks (la versión empaquetada, con tests), no una dependencia. Por eso en este notebook reimplementamos inline, con comentarios, la lógica de `src/llm_lab/utils/hardware.py` y `seed.py`.

**Pasos en Colab:**

1. `File → Open notebook → GitHub` y pegar la URL del repo (o usar el badge de arriba).
2. Activar GPU: `Runtime → Change runtime type → T4 GPU` (o A100/L4 si tienes Pro).
3. Ejecutar las celdas en orden. La celda de setup instala **solo lo que falta**.

**Pasos en local:**

```bash
git clone <repo> && cd llm
python -m venv .venv && source .venv/bin/activate
pip install -e ".[dev]"
jupyter lab notebooks/
```


In [ ]:
# ─── Celda de setup ──────────────────────────────────────────────────
# 1) Detecta si corremos dentro de Google Colab.
# 2) Instala SOLO las dependencias que falten, con `pip install -q`.
#
# IMPORTANTE: en Colab, torch / numpy / matplotlib YA vienen
# preinstalados con versiones compiladas contra el CUDA del runtime.
# Reinstalarlos (p. ej. `pip install torch`) puede romper el entorno.
# Por eso instalamos de forma *condicional*: si el módulo ya se puede
# importar, no tocamos nada.
import importlib.util
import subprocess
import sys

# La forma robusta de detectar Colab: el paquete `google.colab` solo
# existe dentro del runtime de Colab.
IN_COLAB = importlib.util.find_spec("google.colab") is not None
print("Entorno detectado:", "Google Colab" if IN_COLAB else "local / otro")


def ensure_installed(module_name, pip_name=None):
    """Instala `pip_name` solo si `module_name` no es importable."""
    if importlib.util.find_spec(module_name) is None:
        pip_name = pip_name or module_name
        print("Instalando", pip_name, "...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            check=True,
        )
    else:
        print("OK:", module_name, "ya está instalado (no se reinstala)")


# Dependencias mínimas de este notebook. En Colab todas ya están;
# en un entorno local limpio se instalan una sola vez.
ensure_installed("torch")       # núcleo de deep learning
ensure_installed("numpy")
ensure_installed("matplotlib")
ensure_installed("seaborn")


In [ ]:
# ─── Imports y configuración de gráficos ────────────────────────────
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

# Estilo de gráficos consistente en todo el lab
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("NumPy   :", np.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("MPS  disponible:", torch.backends.mps.is_available())


## 2. Detección automática de hardware

PyTorch puede ejecutar el mismo código en tres *backends*, y los probamos siempre en este orden de preferencia:

$$\text{CUDA} \;>\; \text{MPS} \;>\; \text{CPU}$$

| Backend | Qué es | Por qué ese orden |
|---|---|---|
| **CUDA** | GPUs NVIDIA (T4, L4, A100…) | Miles de núcleos + Tensor Cores: el matmul, la operación dominante en un transformer, es 10–100× más rápido que en CPU |
| **MPS** | *Metal Performance Shaders*, la GPU de Apple Silicon (M1/M2/M3) | Acelera bien, pero no todos los kernels de PyTorch están implementados (algunos caen a CPU silenciosamente) |
| **CPU** | Fallback universal | Siempre funciona; suficiente para los demos pequeños del lab |

Además del *device*, necesitamos saber **tres propiedades** que condicionan todo el entrenamiento:

1. **VRAM total** — `torch.cuda.get_device_properties(0).total_memory`: define qué tamaño de modelo cabe (sección 4).
2. **Soporte de `bfloat16`** — `torch.cuda.is_bf16_supported()`: las GPUs Ampere+ (A100, L4) lo soportan; la T4 (Turing) **no**, y eso cambia la estrategia de precisión mixta (sección 3).
3. **Si estamos en Colab** — para decidir si montamos Google Drive (sección 9).

La siguiente celda replica, inline y comentada, la lógica de `src/llm_lab/utils/hardware.py`.


In [ ]:
# ─── Detección de hardware (espejo de src/llm_lab/utils/hardware.py) ─


def get_device():
    """Devuelve el mejor device disponible: cuda > mps > cpu."""
    if torch.cuda.is_available():          # 1º: GPU NVIDIA
        return torch.device("cuda")
    if torch.backends.mps.is_available():  # 2º: GPU Apple Silicon
        return torch.device("mps")
    return torch.device("cpu")             # 3º: fallback universal


def detect_hardware(verbose=True):
    """Inspecciona el hardware y recomienda perfil y precisión.

    Reglas de decisión del lab:
    - GPU con >= 24 GB y soporte bf16 (A100, L4) -> perfil "large", bf16
    - GPU con  < 24 GB (T4 de Colab Free, 16 GB) -> perfil "small", fp16
    - MPS (Apple Silicon) o CPU                  -> perfil "small", fp32

    Devuelve un dict con el snapshot completo del hardware.
    """
    notes = []  # advertencias acumuladas para el usuario

    if torch.cuda.is_available():
        # `get_device_properties(0)` devuelve un objeto con los datos
        # físicos de la GPU 0: nombre, memoria total, compute capability...
        props = torch.cuda.get_device_properties(0)

        # total_memory viene en BYTES -> lo pasamos a GB (1 GB = 1024^3 B)
        vram_gb = props.total_memory / 1024**3

        # bf16 requiere arquitectura Ampere o superior (compute capability
        # >= 8.0). PyTorch lo verifica por nosotros:
        bf16 = torch.cuda.is_bf16_supported()

        # Decisión de perfil: LARGE solo si hay VRAM de sobra Y bf16
        # (las dos cosas a la vez: A100 de 40 GB o L4 de 24 GB).
        profile = "large" if (vram_gb >= 24 and bf16) else "small"
        precision = "bf16" if bf16 else "fp16"

        if not bf16:
            notes.append(
                "GPU sin bf16 (pre-Ampere): usar fp16 con GradScaler "
                "para evitar overflow de gradientes."
            )

        hw = {
            "device": "cuda",
            "device_name": props.name,            # p. ej. "Tesla T4"
            "total_vram_gb": round(vram_gb, 1),
            "supports_bf16": bf16,
            "is_colab": IN_COLAB,
            "recommended_profile": profile,       # "small" | "large"
            "recommended_precision": precision,   # "bf16" | "fp16"
            "notes": notes,
        }

    elif torch.backends.mps.is_available():
        # Apple Silicon: no expone la VRAM (memoria unificada con la RAM)
        # y su soporte de tipos de baja precisión es parcial -> fp32.
        hw = {
            "device": "mps",
            "device_name": "Apple Silicon (MPS)",
            "total_vram_gb": None,
            "supports_bf16": False,
            "is_colab": False,
            "recommended_profile": "small",
            "recommended_precision": "fp32",
            "notes": ["MPS no soporta todos los kernels; algunos ops caen a CPU."],
        }

    else:
        # Sin GPU: todo es posible pero lento. Versiones SMALL siempre.
        hw = {
            "device": "cpu",
            "device_name": "CPU",
            "total_vram_gb": None,
            "supports_bf16": False,
            "is_colab": IN_COLAB,
            "recommended_profile": "small",
            "recommended_precision": "fp32",
            "notes": ["Sin GPU: usar versiones SMALL y reducir max_steps."],
        }

    if verbose:
        print("🖥️  Device:", hw["device"], "(" + hw["device_name"] + ")")
        if hw["total_vram_gb"] is not None:
            print("💾 VRAM total:", hw["total_vram_gb"], "GB | bf16:", hw["supports_bf16"])
        print("📍 Colab:", hw["is_colab"])
        print("✅ Perfil recomendado:", hw["recommended_profile"].upper(),
              "(" + hw["recommended_precision"] + ")")
        for note in hw["notes"]:
            print("⚠️ ", note)

    return hw


In [ ]:
# ─── Ejecutar la detección ───────────────────────────────────────────
HW = detect_hardware(verbose=True)

# `device` lo usaremos en TODO el notebook para mover tensores:
#   x = torch.randn(...).to(device)
device = get_device()
print()
print("Device activo para este notebook:", device)


## 3. Limitaciones de Colab

Colab regala GPUs, pero con condiciones. Conocerlas **antes** de entrenar evita perder horas de cómputo. Las seis que afectan directamente a este lab:

1. **Desconexiones.** Las sesiones Free se cortan a las ~12 h como máximo, y mucho antes si detectan inactividad. Un entrenamiento que no guarda progreso pierde todo. **Mitigación:** *checkpointing* frecuente (cada `eval_every_steps`) hacia Google Drive; todos los loops de entrenamiento de los notebooks 06–07 son reanudables (guardan modelo + optimizador + scheduler + step + estado RNG).

2. **Sin persistencia de disco.** El filesystem `/content` se borra al cerrar la sesión. **Mitigación:** `drive.mount()` (sección 9) para checkpoints y corpus procesado.

3. **fp16 en T4, no bf16.** La T4 (arquitectura Turing, 2018) es anterior a Ampere y solo acelera `float16`, cuyo rango dinámico es minúsculo. Esto obliga a usar *loss scaling* dinámico (`GradScaler`). Lo desarrollamos abajo porque es la limitación más sutil.

4. **RAM de sistema limitada (~12 GB en Free).** Cargar un corpus grande con `pd.read_csv` o listas de Python revienta la RAM (¡la RAM, no la VRAM!). **Mitigación:** HuggingFace `datasets` usa Apache Arrow *memory-mapped*: los datos viven en disco y el SO pagina solo lo necesario.

5. **Cuotas de GPU opacas.** Google no publica los límites; el uso intensivo agota la cuota Free por días. **Mitigación:** los modos SMALL están calibrados para entrenar en < 1 h de T4.

6. **Versiones preinstaladas.** Colab fija torch/numpy compilados contra su CUDA. Instalar otras versiones rompe el runtime — por eso nuestra celda de setup **nunca reinstala torch**.


### 3.1 Las tres GPUs de Colab, comparadas

| | **T4** (Free) | **L4** (Pro) | **A100 40GB** (Pro/Pro+) |
|---|---|---|---|
| VRAM | 16 GB | 24 GB | 40 GB |
| Arquitectura | Turing (2018) | Ada (2023) | Ampere (2020) |
| Compute capability | 7.5 | 8.9 | 8.0 |
| bf16 | ❌ (solo fp16) | ✅ | ✅ |
| TF32 | ❌ | ✅ | ✅ |
| FP16 TFLOPS (pico) | ~65 | ~121 | ~312 |
| Flash Attention | ⚠️ limitado | ✅ | ✅ |
| **Perfil del lab** | **SMALL** | LARGE | **LARGE** |

La columna que más importa para *qué modelo cabe* es la VRAM; la que más importa para *cómo entrenarlo* es bf16. Veamos por qué.


### 3.2 fp16 vs bf16: el mismo número de bits, repartido distinto

Un número en punto flotante se compone de **signo**, **exponente** (controla el rango dinámico) y **mantisa** (controla la precisión). Los tres formatos relevantes:

| Formato | Bits | Exponente | Mantisa | Rango máx. | Precisión relativa ($\epsilon$) |
|---|---|---|---|---|---|
| `float32` | 32 | 8 | 23 | $\approx 3.4 \times 10^{38}$ | $\approx 1.2 \times 10^{-7}$ |
| `float16` | 16 | **5** | 10 | $\mathbf{65\,504}$ | $\approx 9.8 \times 10^{-4}$ |
| `bfloat16` | 16 | **8** | 7 | $\approx 3.4 \times 10^{38}$ | $\approx 7.8 \times 10^{-3}$ |

La idea clave: **bf16 copia el exponente de fp32** (8 bits), así que tiene el *mismo rango dinámico* que fp32 — nunca hace overflow donde fp32 no lo haría — a cambio de perder precisión (solo 7 bits de mantisa).

En cambio fp16, con 5 bits de exponente, satura en $65\,504$: cualquier gradiente o activación que supere ese valor se convierte en `inf`, y un solo `inf` propagado por backprop arruina el paso de entrenamiento. La solución clásica es el **loss scaling**: multiplicar la loss por un factor $s$ antes del backward, de modo que por linealidad del gradiente

$$\nabla_\theta (s \cdot \mathcal{L}) = s \cdot \nabla_\theta \mathcal{L}$$

los gradientes pequeños no se vayan a cero (*underflow*) ni los grandes a `inf`, y dividir por $s$ antes de aplicar el optimizador. `torch.cuda.amp.GradScaler` ajusta $s$ dinámicamente (lo veremos en acción en el notebook 06).

Verifiquémoslo numéricamente — esta celda corre incluso sin GPU, porque los *tipos* existen en CPU:


In [ ]:
# ─── Demostración: rango y precisión de fp32 / fp16 / bf16 ──────────
# torch.finfo expone los límites de cada tipo de punto flotante.
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    info = torch.finfo(dtype)
    print("{:<10s} max={:<12.4g} min_normal={:<12.4g} eps={:.3g}".format(
        str(dtype).replace("torch.", ""), info.max, info.tiny, info.eps))

print()

# 1) OVERFLOW: 70 000 cabe en fp32 y bf16, pero NO en fp16 (max 65 504)
x = torch.tensor(70000.0)                       # escalar fp32
print("70000 en fp16 :", x.to(torch.float16).item(), " <- ¡overflow! (inf)")
print("70000 en bf16 :", x.to(torch.bfloat16).item(), " <- sin overflow (redondeado)")

print()

# 2) PRECISIÓN: bf16 solo tiene 7 bits de mantisa -> redondea agresivo
y = torch.tensor(1.001)
print("1.001 en fp16 :", y.to(torch.float16).item())   # conserva ~3 decimales
print("1.001 en bf16 :", y.to(torch.bfloat16).item())  # el .001 se pierde

# Verificación: el overflow de fp16 es real
assert torch.isinf(x.to(torch.float16))
assert not torch.isinf(x.to(torch.bfloat16))
print()
print("Conclusión: fp16 = más precisión, menos rango (necesita GradScaler);")
print("            bf16 = más rango, menos precisión (entrena estable sin scaler).")


## 4. Presupuesto de VRAM

### 4.1 La regla de los ~16 bytes/parámetro

¿Cuánta VRAM consume **entrenar** un modelo de $N$ parámetros con AdamW y precisión mixta? Mucho más que los pesos. Por cada parámetro hay que guardar:

| Componente | Bytes/parám | Comentario |
|------------|:-----------:|------------|
| Pesos (fp16/bf16) | 2 | la copia con la que se computa el forward/backward |
| Pesos *master* (fp32) | 4 | copia de alta precisión que mantiene el optimizador |
| Gradientes | 2 | en la precisión de cómputo |
| AdamW: momento $m$ | 4 | media móvil del gradiente, en fp32 |
| AdamW: varianza $v$ | 4 | media móvil del gradiente², en fp32 |
| **Total estados** | **~16** | |

$$\text{VRAM}_{\text{estados}} \approx 16 \, N \ \text{bytes}$$

A esto se suma la **memoria de activaciones**, que no depende de $N$ sino del batch y la secuencia: cada capa guarda sus salidas intermedias para el backward, con costo $O(B \cdot T \cdot d)$ por capa, y la matriz de attention materializada cuesta $O(B \cdot h \cdot T^2)$ — el término cuadrático en $T$ que motiva Flash Attention (notebook 05, §13). En secuencias largas, **las activaciones dominan** sobre los estados del modelo.

### 4.2 Cuántos parámetros tienen nuestros modelos

Para un transformer decoder-only sin biases, con embeddings atados (`tie_embeddings: true`: la capa de salida reutiliza $W_{emb}$):

$$N \approx \underbrace{V \cdot d}_{\text{embeddings}} + L \cdot \big( \underbrace{4d^2}_{W_Q, W_K, W_V, W_O} + \underbrace{2 \, d \cdot d_{ff}}_{\text{FFN}} + \underbrace{4d}_{\text{2 LayerNorm}} \big)$$

donde $V$ = vocabulario, $d$ = `d_model`, $L$ = número de capas, $d_{ff}$ = dimensión interna del FFN.

> **Nota sobre convenciones:** cuando los papers de *scaling laws* (Kaplan et al., 2020) hablan del "tamaño" de un modelo suelen contar solo los **parámetros del cuerpo transformer** (sin embeddings), porque son los que escalan con el cómputo. Nuestro LARGE tiene ~85 M de parámetros *de cuerpo* (+ ~25 M de embeddings con vocab 32k). La celda siguiente calcula ambos.


In [ ]:
# ─── Conteo de parámetros de los modelos SMALL y LARGE del lab ──────
# Hiperparámetros tomados de configs/small.yaml y configs/large.yaml


def count_params(vocab_size, d_model, n_layers, d_ff, tie_embeddings=True):
    """Cuenta parámetros de un decoder-only transformer (sin biases)."""
    embeddings = vocab_size * d_model        # W_emb: (V, d_model)

    # Por cada bloque transformer:
    attn = 4 * d_model * d_model             # W_q, W_k, W_v, W_o: 4 de (d, d)
    ffn = 2 * d_model * d_ff                 # W_1: (d, d_ff) y W_2: (d_ff, d)
    norms = 2 * 2 * d_model                  # 2 LayerNorm x (gamma, beta)
    block = attn + ffn + norms

    final_norm = 2 * d_model                 # LayerNorm final
    # Con tie_embeddings la capa de salida (lm_head) REUTILIZA W_emb -> 0 extra
    lm_head = 0 if tie_embeddings else vocab_size * d_model

    body = n_layers * block + final_norm     # "cuerpo" transformer
    total = embeddings + body + lm_head
    return {"embeddings": embeddings, "body": body, "total": total}


# Definición de los dos perfiles (espejo de configs/*.yaml)
MODEL_SPECS = {
    "SMALL": dict(vocab_size=8000, d_model=256, n_layers=4, d_ff=1024),
    "LARGE": dict(vocab_size=32000, d_model=768, n_layers=12, d_ff=3072),
}

PARAM_COUNTS = {}
for name, spec in MODEL_SPECS.items():
    p = count_params(**spec)
    PARAM_COUNTS[name] = p
    print("{:<6s} embeddings: {:>6.2f} M | cuerpo: {:>6.2f} M | TOTAL: {:>6.2f} M".format(
        name, p["embeddings"] / 1e6, p["body"] / 1e6, p["total"] / 1e6))

# Verificación de orden de magnitud contra la documentación del lab:
assert 3e6 < PARAM_COUNTS["SMALL"]["total"] < 7e6      # "~4-5 M"
assert 80e6 < PARAM_COUNTS["LARGE"]["body"] < 90e6     # "~85 M" de cuerpo
print()
print("El LARGE tiene ~85 M de parámetros de cuerpo (la cifra de los docs)")
print("y ~110 M totales si contamos los embeddings de vocab 32k.")


In [ ]:
# ─── Presupuesto de VRAM: estados de entrenamiento + activaciones ───
# Regla: ~16 bytes/parámetro con AdamW + precisión mixta
BYTES_WEIGHTS = 2        # pesos en fp16/bf16
BYTES_GRADS = 2          # gradientes en fp16/bf16
BYTES_OPTIMIZER = 12     # master fp32 (4) + momento m (4) + varianza v (4)

# Batch y secuencia de entrenamiento de cada perfil (configs/*.yaml)
TRAIN_SPECS = {
    "SMALL": dict(B=32, T=256, n_heads=4),
    "LARGE": dict(B=64, T=1024, n_heads=12),
}


def estimate_activations_gb(B, T, d_model, n_layers, n_heads, bytes_per_el=2):
    """Estimación de ORDEN DE MAGNITUD de la memoria de activaciones.

    - Término lineal: ~10 tensores intermedios de (B, T, d) por capa
      (salidas de proyecciones, FFN, norms, residuales...).
    - Término cuadrático: la matriz de attention (B, h, T, T) por capa,
      SI se materializa (Flash Attention la evita -> notebook 05 §13).
    """
    linear_elems = 10 * B * T * d_model        # O(B·T·d) por capa
    attn_elems = B * n_heads * T * T           # O(B·h·T²) por capa
    total_elems = n_layers * (linear_elems + attn_elems)
    return total_elems * bytes_per_el / 1024**3


# Construimos el desglose en GB para cada perfil
breakdown = {}   # {perfil: [pesos, gradientes, optimizador, activaciones]}
for name in ["SMALL", "LARGE"]:
    n_params = PARAM_COUNTS[name]["total"]
    spec = MODEL_SPECS[name]
    tspec = TRAIN_SPECS[name]
    weights_gb = n_params * BYTES_WEIGHTS / 1024**3
    grads_gb = n_params * BYTES_GRADS / 1024**3
    optim_gb = n_params * BYTES_OPTIMIZER / 1024**3
    acts_gb = estimate_activations_gb(
        B=tspec["B"], T=tspec["T"], d_model=spec["d_model"],
        n_layers=spec["n_layers"], n_heads=tspec["n_heads"])
    breakdown[name] = [weights_gb, grads_gb, optim_gb, acts_gb]
    total = sum(breakdown[name])
    print("{:<6s} pesos={:.2f}  grads={:.2f}  optim={:.2f}  activs~{:.1f}  ->  TOTAL ~{:.1f} GB".format(
        name, weights_gb, grads_gb, optim_gb, acts_gb, total))

# ─── Gráfico de barras apiladas del desglose ─────────────────────────
component_labels = ["Pesos (2 B/parám)", "Gradientes (2 B/parám)",
                    "Optimizador (12 B/parám)", "Activaciones (estimación)"]
component_colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(2)                       # (2,) una barra por perfil
bottom = np.zeros(2)                   # (2,) acumulador del apilado
for i, label in enumerate(component_labels):
    vals = np.array([breakdown["SMALL"][i], breakdown["LARGE"][i]])  # (2,)
    ax.bar(x, vals, bottom=bottom, label=label,
           color=component_colors[i], width=0.5)
    bottom += vals

# Líneas de referencia: la VRAM de cada GPU de Colab
for vram, gpu in [(16, "T4"), (24, "L4"), (40, "A100")]:
    ax.axhline(vram, ls="--", lw=1, color="gray", alpha=0.7)
    ax.text(1.42, vram + 0.4, gpu + " (" + str(vram) + " GB)",
            fontsize=9, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(["SMALL (~5 M params)", "LARGE (~110 M params)"])
ax.set_ylabel("VRAM (GB)")
ax.set_title("Presupuesto de VRAM de entrenamiento (AdamW + precisión mixta)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print()
print("Lectura del gráfico: los ESTADOS del modelo son baratos (16 bytes/parám,")
print("~1.7 GB incluso para LARGE), pero las ACTIVACIONES con T=1024 y B=64")
print("dominan por completo. De ahí Flash Attention, gradient checkpointing")
print("y micro-batches con gradient accumulation (notebooks 05 y 06).")


## 5. Seeds y reproducibilidad

### Las 4 fuentes de aleatoriedad

Un experimento de ML usa generadores pseudoaleatorios en **al menos cuatro lugares independientes** — fijar uno solo NO garantiza reproducibilidad:

| # | Fuente | Quién la usa | Cómo se fija |
|---|--------|--------------|---------------|
| 1 | `random` de Python | shuffling de listas, sampling simple | `random.seed(s)` |
| 2 | NumPy | inicializaciones, sampling de datasets | `np.random.seed(s)` |
| 3 | PyTorch CPU | **inicialización de pesos**, dropout | `torch.manual_seed(s)` |
| 4 | PyTorch CUDA + cuDNN | kernels en GPU, autotuner de cuDNN | `torch.cuda.manual_seed_all(s)` + flags de cuDNN |

La cuarta es la más traicionera: aunque fijes la seed de CUDA, **cuDNN elige el algoritmo por *benchmarking*** (`torch.backends.cudnn.benchmark = True`), y distintos algoritmos suman los flotantes en distinto orden. Como la suma en punto flotante **no es asociativa** —

$$(a + b) + c \neq a + (b + c) \quad \text{en general, con redondeo finito}$$

— el mismo cómputo puede dar bits distintos entre ejecuciones. Para reproducibilidad exacta se fuerza `cudnn.deterministic = True` y `cudnn.benchmark = False` (cuesta ~5–10 % de throughput; para benchmarks de velocidad se desactiva, como haremos en la sección 7).

También fijamos `PYTHONHASHSEED`, que afecta el orden de iteración de sets/dicts entre procesos.

Primero, la demostración de que **sin seed no hay reproducibilidad**:


In [ ]:
# ─── Demostración 1: SIN seed, cada ejecución es distinta ───────────
# Pedimos dos veces "los mismos" números aleatorios... y no lo son,
# porque el generador avanza su estado interno con cada llamada.
draw_a = torch.randn(5)    # (5,) muestras de N(0, 1)
draw_b = torch.randn(5)    # (5,) otras 5 muestras: el RNG ya avanzó

print("Tirada A:", np.round(draw_a.numpy(), 4))
print("Tirada B:", np.round(draw_b.numpy(), 4))

# Verificación: son distintas (la probabilidad de colisión es ~0)
assert not torch.allclose(draw_a, draw_b)
print()
print("Sin seed fija, dos 'inicializaciones de pesos' idénticas en el código")
print("producen modelos DISTINTOS -> experimentos imposibles de comparar.")


In [ ]:
# ─── Demostración 2: CON seed, reproducibilidad exacta ──────────────
# Espejo de src/llm_lab/utils/seed.py
SEED = 42


def set_seed(seed=42, deterministic_cudnn=True):
    """Fija la seed en las 4 fuentes de aleatoriedad."""
    random.seed(seed)                  # 1) random de Python
    np.random.seed(seed)               # 2) NumPy
    torch.manual_seed(seed)            # 3) PyTorch CPU
    torch.cuda.manual_seed_all(seed)   # 4) PyTorch CUDA (todas las GPUs;
                                       #    no-op silencioso si no hay GPU)

    # El hash de strings afecta el orden de sets/dicts entre procesos
    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic_cudnn and torch.cuda.is_available():
        # benchmark=True deja que cuDNN elija el kernel más rápido por
        # heurística (no determinista entre ejecuciones) -> lo apagamos
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    return seed


# Fijamos la seed, muestreamos de las 3 librerías... y repetimos:
set_seed(SEED)
py_1 = random.random()
np_1 = np.random.rand(3)               # (3,)
torch_1 = torch.randn(3)               # (3,)

set_seed(SEED)                         # <- reinicia los 4 generadores
py_2 = random.random()
np_2 = np.random.rand(3)               # (3,)
torch_2 = torch.randn(3)               # (3,)

print("Python random :", round(py_1, 6), "==", round(py_2, 6))
print("NumPy         :", np.round(np_1, 4), "==", np.round(np_2, 4))
print("PyTorch       :", np.round(torch_1.numpy(), 4), "==",
      np.round(torch_2.numpy(), 4))

# Verificación exacta (bit a bit) de las tres fuentes
assert py_1 == py_2
assert np.allclose(np_1, np_2)
assert torch.allclose(torch_1, torch_2)
print()
print("✅ Reproducibilidad verificada. SEED =", SEED, "queda fijada para el lab.")


## 6. Selección de perfil SMALL/LARGE

Todo el laboratorio se ejecuta en uno de dos perfiles, definidos en `configs/small.yaml` y `configs/large.yaml`:

| | **SMALL** | **LARGE** |
|---|---|---|
| Hardware objetivo | T4 (16 GB) / CPU / MPS | A100 (40 GB) / L4 (24 GB) |
| Parámetros | ~4–5 M | ~85 M (cuerpo) |
| `d_model` / capas / heads | 256 / 4 / 4 | 768 / 12 / 12 |
| Contexto (`max_seq_len`) | 256 | 1024 |
| Vocabulario | 8 000 | 32 000 |
| Precisión mixta | fp16 (+ GradScaler) | bf16 |
| Batch (efectivo) | 32 | 64 × 4 acum. = 256 |

La decisión es automática a partir del hardware detectado en la sección 2, pero **siempre puedes forzarla a mano** (por ejemplo, SMALL en una A100 para iterar rápido):

```mermaid
flowchart LR
    A[detect_hardware] --> B{VRAM >= 24 GB<br/>y bf16?}
    B -- sí --> C[LARGE · bf16]
    B -- no --> D{GPU CUDA?}
    D -- sí --> E[SMALL · fp16]
    D -- no --> F[SMALL · fp32<br/>CPU / MPS]
```

*(Descripción textual del diagrama, para Colab: si la GPU tiene ≥ 24 GB de VRAM y soporta bf16 → perfil LARGE con bf16; si hay GPU CUDA pero sin esas dos condiciones → SMALL con fp16; sin GPU CUDA → SMALL con fp32 en CPU/MPS.)*

La celda siguiente es la **celda de configuración** estándar que vas a ver, con variaciones, en todos los notebooks 01–08.


In [ ]:
# ─── Celda de configuración SMALL/LARGE ──────────────────────────────
# "auto" usa la recomendación del hardware (sección 2).
# Para forzar un perfil a mano: PROFILE = "small" o PROFILE = "large".
PROFILE = "auto"

if PROFILE == "auto":
    PROFILE = HW["recommended_profile"]
assert PROFILE in ("small", "large"), "PROFILE debe ser 'auto', 'small' o 'large'"

# Hiperparámetros de ambos perfiles (espejo de configs/small.yaml y large.yaml)
CONFIG_BY_PROFILE = {
    "small": {
        "vocab_size": 8000,
        "d_model": 256,
        "n_layers": 4,
        "n_heads": 4,
        "d_ff": 1024,
        "max_seq_len": 256,
        "batch_size": 32,
        "gradient_accumulation_steps": 1,
        "max_steps": 3000,
        "lr": 3.0e-4,
        "mixed_precision": "fp16",   # la T4 no soporta bf16
    },
    "large": {
        "vocab_size": 32000,
        "d_model": 768,
        "n_layers": 12,
        "n_heads": 12,
        "d_ff": 3072,
        "max_seq_len": 1024,
        "batch_size": 64,
        "gradient_accumulation_steps": 4,   # batch efectivo = 256
        "max_steps": 20000,
        "lr": 6.0e-4,
        "mixed_precision": "bf16",   # A100/L4: bfloat16 nativo
    },
}

CONFIG = CONFIG_BY_PROFILE[PROFILE]

print("Perfil activo :", PROFILE.upper())
print("Device        :", device)
for key, value in CONFIG.items():
    print("  {:<30s} {}".format(key, value))


## 7. Benchmark de matmul

La multiplicación de matrices es **la** operación de un transformer (attention y FFN son, en esencia, matmuls gigantes). Medir cuántos FLOPS reales entrega tu hardware te da una vara para estimar tiempos de entrenamiento.

Multiplicar $A \in \mathbb{R}^{n \times n}$ por $B \in \mathbb{R}^{n \times n}$ cuesta

$$\text{FLOPs} = 2n^3$$

(cada uno de los $n^2$ elementos de salida requiere $n$ multiplicaciones y $n$ sumas). Si la operación tarda $t$ segundos, el rendimiento medido es $2n^3 / t$ FLOPS.

**Ejemplo numérico verificable a mano:** con $n = 1024$, FLOPs $= 2 \cdot 1024^3 \approx 2.15 \times 10^9$. Si el matmul tarda 1 ms, el hardware entrega $2.15 \times 10^9 / 10^{-3} = 2.15$ TFLOPS.

**Detalle crítico de medición en GPU:** los kernels CUDA son *asíncronos* — `a @ b` retorna inmediatamente y el cómputo sigue en la GPU. Sin `torch.cuda.synchronize()` antes de parar el reloj, medirías el tiempo de *lanzar* el kernel, no de ejecutarlo.

Esperamos ver que **fp16 rinde varias veces más que fp32** en GPU: los Tensor Cores (T4: ~65 TFLOPS fp16 vs ~8 fp32; A100: ~312 vs ~19.5) solo se activan con tipos de 16 bits (o TF32 en Ampere+).


In [ ]:
# ─── Benchmark de matmul: TFLOPS medidos en fp32 y fp16 ─────────────


def benchmark_matmul(n, dtype, device, n_iters=10, n_warmup=3):
    """Mide TFLOPS de (n, n) @ (n, n) en el dtype y device dados."""
    a = torch.randn(n, n, dtype=dtype, device=device)   # (n, n)
    b = torch.randn(n, n, dtype=dtype, device=device)   # (n, n)

    # Calentamiento: la primera llamada compila/elige kernels y no es
    # representativa del rendimiento sostenido.
    for _ in range(n_warmup):
        _ = a @ b                                        # (n, n)

    if device.type == "cuda":
        torch.cuda.synchronize()    # espera a que termine el warmup

    t0 = time.perf_counter()
    for _ in range(n_iters):
        _ = a @ b                                        # (n, n)
    if device.type == "cuda":
        torch.cuda.synchronize()    # espera a que terminen los kernels
    elapsed = (time.perf_counter() - t0) / n_iters

    flops = 2 * n**3                # 2n^3 FLOPs por matmul
    return flops / elapsed / 1e12   # -> TFLOPS


# Para el benchmark queremos VELOCIDAD, no determinismo (sección 5):
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# Tamaños adaptados al hardware: en CPU los matmuls grandes tardan minutos
if device.type == "cuda":
    sizes = [1024, 2048, 4096]
    bench_dtypes = [torch.float32, torch.float16]  # fp16 -> Tensor Cores
else:
    sizes = [256, 512, 1024]
    bench_dtypes = [torch.float32]                 # fp16 en CPU no acelera
    print("Sin GPU CUDA: benchmark reducido, solo fp32.")

results = {}    # {dtype: lista de TFLOPS, una por tamaño}
for dtype in bench_dtypes:
    results[dtype] = []
    for n in sizes:
        tflops = benchmark_matmul(n, dtype, device)
        results[dtype].append(tflops)
        print("n={:<5d} {:<10s} {:>8.2f} TFLOPS".format(
            n, str(dtype).replace("torch.", ""), tflops))

# ─── Gráfico: TFLOPS por tamaño y precisión ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
width = 0.35
x = np.arange(len(sizes))                         # (n_sizes,)
for i, dtype in enumerate(bench_dtypes):
    offset = (i - (len(bench_dtypes) - 1) / 2) * width
    ax.bar(x + offset, results[dtype], width,
           label=str(dtype).replace("torch.", ""))
ax.set_xticks(x)
ax.set_xticklabels(["n = " + str(n) for n in sizes])
ax.set_ylabel("TFLOPS medidos")
ax.set_title("Benchmark de matmul en " + str(device) +
             " (" + HW["device_name"] + ")")
ax.legend()
plt.tight_layout()
plt.show()

print()
print("Observa: (1) los matmuls grandes rinden más (amortizan el overhead de")
print("lanzar kernels); (2) en GPU, fp16 multiplica el rendimiento porque")
print("activa los Tensor Cores. El pico teórico casi nunca se alcanza.")


## 8. Monitoreo de VRAM

Durante los entrenamientos (notebooks 06–07) vigilaremos la VRAM constantemente. PyTorch expone tres contadores clave:

| Función | Qué mide |
|---|---|
| `torch.cuda.memory_allocated()` | Bytes **ocupados por tensores vivos** ahora mismo |
| `torch.cuda.memory_reserved()` | Bytes que el *caching allocator* de PyTorch **reservó** al driver (≥ allocated: PyTorch retiene bloques liberados para reusarlos sin pedir memoria de nuevo) |
| `torch.cuda.max_memory_allocated()` | **Pico** histórico de allocated — el número que anticipa los `CUDA out of memory` |

La diferencia `reserved - allocated` es memoria "fantasma": libre para PyTorch pero invisible para `nvidia-smi`, que reporta lo reservado. Por eso `nvidia-smi` siempre muestra más uso del que tus tensores explican.


In [ ]:
# ─── Snapshot de VRAM (espejo de gpu_memory_snapshot en hardware.py) ─


def gpu_memory_snapshot():
    """Devuelve el uso actual de VRAM en GB (solo CUDA)."""
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 3),
        "reserved_gb": round(torch.cuda.memory_reserved() / 1024**3, 3),
        "max_allocated_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }


if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()    # reinicia el contador de pico
    print("Antes de alocar  :", gpu_memory_snapshot())

    # Alocamos un tensor de 4096x4096 en fp32:
    # 4096 * 4096 * 4 bytes = 64 MiB exactos -> verificable en el snapshot
    big = torch.randn(4096, 4096, device=device)   # (4096, 4096)
    print("Con tensor 64 MiB:", gpu_memory_snapshot())

    del big                          # el tensor muere...
    torch.cuda.empty_cache()         # ...y devolvemos la cache al driver
    print("Tras liberar     :", gpu_memory_snapshot())
    print()
    print("Nota: max_allocated_gb conserva el PICO aunque ya liberamos --")
    print("es el contador que hay que vigilar para anticipar OOMs.")
else:
    print("Sin GPU CUDA: el monitoreo de VRAM no aplica en", device.type)
    print("(En MPS existe torch.mps.current_allocated_memory() como análogo.)")


## 9. Google Drive para checkpoints

En Colab, **todo lo que no esté en Drive se pierde** al cerrar la sesión (limitación 2 de la sección 3). La convención del lab:

- Checkpoints → `MyDrive/llm_lab/checkpoints/`
- Corpus procesado → `MyDrive/llm_lab/datasets/`

La celda siguiente es **condicional**: monta Drive solo si estamos en Colab (pedirá autorización OAuth la primera vez); en local usa el directorio `checkpoints/` del repo.


In [ ]:
# ─── Montaje condicional de Google Drive ─────────────────────────────
if IN_COLAB:
    from google.colab import drive

    # Pide autorización la primera vez; luego queda montado en /content/drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/llm_lab/checkpoints"
else:
    # En local: carpeta del repo (no versionada, ver .gitignore)
    CHECKPOINT_DIR = os.path.join("..", "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Directorio de checkpoints:", os.path.abspath(CHECKPOINT_DIR))

# Prueba de escritura: si esto falla, mejor enterarse AHORA y no en el
# paso 2999 de un entrenamiento de 3000 pasos.
probe_path = os.path.join(CHECKPOINT_DIR, ".write_probe")
with open(probe_path, "w") as f:
    f.write("ok")
os.remove(probe_path)
print("✅ Escritura verificada: los checkpoints sobrevivirán a la sesión.")


## 10. Resumen

Lo que dejamos configurado y aprendido:

1. **Entorno**: detección de Colab e instalación condicional de dependencias (sin tocar el torch preinstalado).
2. **Hardware**: jerarquía CUDA > MPS > CPU; las propiedades que importan son la VRAM total y el soporte de bf16 (`get_device_properties`, `is_bf16_supported`).
3. **Colab**: 6 limitaciones con sus mitigaciones. La más sutil: la T4 solo tiene fp16 (rango máx. 65 504 → `GradScaler`); A100/L4 tienen bf16 (rango de fp32, entrenamiento estable sin scaler).
4. **VRAM**: entrenar con AdamW + precisión mixta cuesta **~16 bytes/parámetro** en estados (2 pesos + 2 grads + 12 optimizador) **más** activaciones $O(B \cdot T \cdot d)$ y $O(B \cdot h \cdot T^2)$, que dominan en secuencias largas.
5. **Reproducibilidad**: 4 fuentes de aleatoriedad (Python, NumPy, torch CPU, CUDA/cuDNN); demostramos no-reproducibilidad sin seed y reproducibilidad exacta con `set_seed(42)`.
6. **Perfil**: `PROFILE` (auto/small/large) y `CONFIG` quedan definidos — el patrón que repiten los notebooks 01–08.
7. **Medición**: TFLOPS reales con matmul (sincronizando CUDA antes de parar el reloj) y monitoreo de VRAM (`memory_allocated` / `memory_reserved` / pico).
8. **Persistencia**: checkpoints hacia Google Drive si Colab, carpeta local si no.

**Siguiente parada** → `01_fundamentos_matematicos.ipynb`: álgebra lineal, tensores, backprop, softmax, cross-entropy y scaling laws.


## 11. Ejercicios propuestos

1. **(Fácil)** Calcula a mano cuánta VRAM (solo estados, sin activaciones) necesita entrenar GPT-2 small (124 M de parámetros) con AdamW + precisión mixta. Verifícalo con la regla de los 16 bytes/parámetro: ¿cabrían los estados, junto con activaciones de B=8 y T=1024, en una T4?

2. **(Fácil)** Cambia `PROFILE = "auto"` por `"large"` corriendo en CPU y reejecuta la celda de configuración. ¿Qué hiperparámetros cambian? ¿Por qué sería mala idea entrenar así?

3. **(Media)** Modifica `estimate_activations_gb` para añadir un flag `flash_attention: bool` que elimine el término $O(B \cdot h \cdot T^2)$. ¿Cuánta VRAM ahorra Flash Attention en el perfil LARGE? Regrafica el desglose de barras.

4. **(Media)** En el benchmark de matmul, añade `torch.bfloat16` (si tu GPU lo soporta) y matrices rectangulares $(B \cdot T) \times d$ por $d \times 4d$ con los valores del perfil activo — la forma real del FFN. ¿El rendimiento se parece al del matmul cuadrado?

5. **(Difícil)** La reproducibilidad de la sección 5 es *dentro de la misma máquina*. Investiga `torch.use_deterministic_algorithms(True)` y la variable de entorno `CUBLAS_WORKSPACE_CONFIG`: ¿qué operaciones de un transformer siguen siendo no deterministas entre GPUs distintas y por qué (pista: orden de reducción en punto flotante)?


## 12. Referencias

**Papers**

- Micikevicius et al. (2017). *Mixed Precision Training*. [arXiv:1710.03740](https://arxiv.org/abs/1710.03740) — el paper original del loss scaling y los pesos master en fp32.
- Kalamkar et al. (2019). *A Study of BFLOAT16 for Deep Learning Training*. [arXiv:1905.12322](https://arxiv.org/abs/1905.12322) — por qué bf16 entrena estable sin scaler.
- Rajbhandari et al. (2019). *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models*. [arXiv:1910.02054](https://arxiv.org/abs/1910.02054) — la contabilidad de los 16 bytes/parámetro, formalizada.
- Korthikanti et al. (2022). *Reducing Activation Recomputation in Large Transformer Models*. [arXiv:2205.05198](https://arxiv.org/abs/2205.05198) — fórmulas exactas de memoria de activaciones.
- Dao et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135) — elimina la matriz $O(T^2)$.
- Kaplan et al. (2020). *Scaling Laws for Neural Language Models*. [arXiv:2001.08361](https://arxiv.org/abs/2001.08361) — la convención de contar parámetros sin embeddings.

**Documentación**

- [PyTorch — Reproducibility](https://pytorch.org/docs/stable/notes/randomness.html)
- [PyTorch — CUDA semantics (caching allocator)](https://pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch — Automatic Mixed Precision](https://pytorch.org/docs/stable/amp.html)

**Del repo**

- `docs/05_colab_y_hardware.md` — limitaciones de Colab y presupuestos de VRAM.
- `src/llm_lab/utils/hardware.py` y `src/llm_lab/utils/seed.py` — el código espejo de este notebook.
- `configs/small.yaml` / `configs/large.yaml` — los perfiles completos.
